# MCH–MFT Track Matching NN

Binary classifier NN script based on the XGB implementation

# Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import importlib
import Utils
import optuna
import Evaluation
import shap

from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix, classification_report, accuracy_score, average_precision_score, log_loss, auc
from hipe4ml.tree_handler import TreeHandler
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import softmax
from scipy.stats import ks_2samp
from sklearn.inspection import permutation_importance
import seaborn as sns
from sklearn.calibration import CalibrationDisplay
from matplotlib import cm

# NN additions
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import joblib


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
importlib.reload(Utils)
importlib.reload(Evaluation)


# Load, Format, Engineer Data

In [ ]:
# df = Utils.get_dataframe("OO-LHC25i4_FIXED.root", folder_name="DF_*")
# df = Utils.get_dataframe("Data/PbPbLHC26b13_FIXED.root", folder_name="DF_*")
df = Utils.get_dataframe("Data/PbPbTestdata.root", folder_name="DF_*")
# df = Utils.get_dataframe("Data/OOLHC25i4testing.root", folder_name="DF_*")


# df_TESTDATA = Utils.get_dataframe("Data/PbPbTestdata.root", folder_name="DF_*")
# df_train = Utils.get_dataframe("Data/PbPbWeighted.root", folder_name="DF_*")

In [ ]:
# df_train = Utils.process_dataframe(df_train, makedummies=False)
# df_TESTDATA = Utils.process_dataframe(df_TESTDATA, makedummies=False)

In [ ]:
# np.seterr(all='raise')
df = Utils.process_dataframe(df, makedummies=False)


TARGET = "IsSignal"

GROUP  = "mchID"



In [ ]:
# df_TESTDATA = Utils.subsample(df_TESTDATA, frac = 0.3)


In [ ]:
FEATURES = [f for f in df.columns.tolist() if f not in Utils.NON_TRAINING_FEATURES]
# FEATURES = ['DeltaDirection', 'APullPhi', 'PullPt', 'PtMFT', 'PullTanl',
#        'DeltaR', 'SameSign', 'RelPtDiff', 'CPhiPhiMFT', 'DeltaTanl']
# FEATURES = Utils.FEATURES_OO_UNCORRELATED
# FEATURES = ['DeltaDirection', 'PullPt', 'APullPhi', 'PullTanl', 'PtMFT',
#        'CPhiPhiMFT', 'DeltaR', 'SameSign', 'DeltaEta', 'DeltaTanl',
#        'RelPtDiff', 'CYYMFT', 'C1Pt1PtMFT', 'PullPhi', 'CXXMFT',
#        'C1PtPhiMFT', 'YMCH', 'XMCH', 'DeltaPt', 'ADeltaPhi', 'PullR',
#        'PullX', 'PullY', 'CXYMFT', 'CTglTglMCH', 'DeltaPhi', 'C1PtXMFT']
# SuperSlim Features model

# FEATURES = ['RelPtDiff', 'DeltaDirection', 'SameSign', 'DeltaR', 'PtMFT',
#        'ADeltaPhi', 'CXXMFT', 'CPhiPhiMCH', 'PullPt', 'CYYMFT',
#        'C1Pt1PtMFT', 'CPhiPhiMFT', 'CTglTglMCH', 'C1Pt1PtMCH', 'DeltaPt',
#        'TanlMFT', 'PullR', 'ADeltaX', 'DeltaTanl', 'PullTanl', 'ADeltaY',
#        'C1PtPhiMFT', 'CTglTglMFT', 'etaMFT', 'DeltaEta', 'APullPhi',
#        ]


# FEATURES = ['RelPtDiff', 'SameSign', 'PtMFT', 'PullPt', 'CPhiPhiMFT',
#        'C1Pt1PtMFT', 'CTglTglMCH', 'CPhiPhiMCH', 'TanlMFT', 'CXXMFT',
#        'CYYMFT', 'DeltaDirection', 'DeltaR', 'ADeltaPhi', 'etaMFT',
#        'PullR', 'DeltaPt', 'C1PtPhiMFT', 'DeltaEta', 'ADeltaX',
#        'APullPhi', 'DeltaTanl', 'CYYMCH', 'ADeltaY', 'CTglTglMFT',
#        'PullTanl', 'CXXMCH', 'InvQPtMFT', 'PtMCH', 'C1Pt1PtMCH', 'PullY',
#        'CXYMFT', 'APullX', 'DeltaX', 'APullY', 'DeltaPhi', 'C1PtXMFT',
#        'DeltaY', 'CTglXMCH', 'etaMCH', 'PullX', 'YMCH', 'TanlMCH', 'XMCH',
#        'CPhiXMFT', 'CTglXMFT', 'CPhiYMFT', 'C1PtYMFT', 'CTglPhiMFT',
#        'PullPhi', 'XMFT', 'YMFT', 'CPhiYMCH', 'CTglYMFT', 'PhiMFT',
#        'PhiMCH', 'C1PtTglMCH', 'CTglYMCH', 'C1PtTglMFT', 'CPhiXMCH'] # sLIGHTLY longer list of features we NEED for the pbpb system

In [ ]:
# df = df[(df['PtMCH'] > 0.7) & (df['PtMCH'] < 5.0)] # focus on low 
# df = df[(df['MFTMult'] <750) & (df['MFTMult'] >500)] # focus on low multiplicity events

In [ ]:
# Evaluation.PeekData(df_train)
# Evaluation.PeekData(df_TESTDATA)
Evaluation.PeekData(df)

# Train / Test Split

Split is done **by MCH track group**, not by row, to avoid data leakage  
(candidates from the same MCH track must not appear in both train and test).

In [ ]:
df_train, df_val, df_test = Evaluation.Splitter(df, val_frac=0.1, test_frac = 0.3)
# df_val, df_test = Evaluation.splitter_internal(df_TESTDATA, test_frac=0.8)

# --- Metrics & Verification ---
print(f"Train: {len(df_train):,} pairs ({df_train[GROUP].nunique():,} MCH tracks)")
print(f"Val:   {len(df_val):,} pairs ({df_val[GROUP].nunique():,} MCH tracks)")
print(f"Test:  {len(df_test):,} pairs ({df_test[GROUP].nunique():,} MCH tracks)\n")

print(f"Train positive rate: {df_train[TARGET].mean():.3f}")
print(f"Val   positive rate: {df_val[TARGET].mean():.3f}")
print(f"Test  positive rate: {df_test[TARGET].mean():.3f}\n")

# --- Sample Weight Calculation (Only using Training Distribution) ---
neg = (df_train[TARGET] == 0).sum()
pos = (df_train[TARGET] == 1).sum()
spw = neg / pos
print(f"scale_pos_weight = {spw:.2f} (neg={neg:,}, pos={pos:,})")

# Scaling

In [ ]:
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(df_train[FEATURES])
X_val_scaled  = scaler.transform(df_val[FEATURES])
X_test_scaled  = scaler.transform(df_test[FEATURES])

y_train = df_train[TARGET]
y_validation = df_val[TARGET]
y_test = df_test[TARGET]


# Save scaler alongside model for inference
import joblib
joblib.dump(scaler, "track_matching_scaler.pkl")
print("Scaler fitted and saved.")
print(f"Feature means (sample): {scaler.mean_[:3].round(4)}")
print(f"Feature stds  (sample): {scaler.scale_[:3].round(4)}")

# Define Model and parameters

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
HIDDEN_LAYERS = [32, 32]
DROPOUT       = 0.2
BATCH_SIZE    = 4096
N_EPOCHS      = 50
LR            = 1e-3
PATIENCE      = 3    # early stopping

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# ── Model definition ──────────────────────────────────────────────────────────
class TrackMatchMLP(nn.Module):
    def __init__(self, n_features: int, hidden_layers: list, dropout: float):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden_layers:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))   # single logit output
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)          # (batch,)

# ── Class imbalance weight ────────────────────────────────────────────────────
neg  = (y_train == 0).sum()
pos  = (y_train == 1).sum()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(device)
print(f"pos_weight = {pos_weight.item():.2f}  (neg={neg:,}, pos={pos:,})")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Dataloaders ───────────────────────────────────────────────────────────────
X_tr = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr = torch.tensor(y_train.values, dtype=torch.float32)
X_val = torch.tensor(X_val_scaled,  dtype=torch.float32)
y_val = torch.tensor(y_validation.values,  dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_tr, y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

model_nn = TrackMatchMLP(
    n_features=len(FEATURES),
    hidden_layers=HIDDEN_LAYERS,
    dropout=DROPOUT,
).to(device)

optimiser = torch.optim.Adam(model_nn.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode="min", patience=5, factor=0.5
)

print(f"\nModel architecture:\n{model_nn}")
n_params = sum(p.numel() for p in model_nn.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

# ── Training loop ─────────────────────────────────────────────────────────────
train_losses, test_losses = [], []
best_test_loss = np.inf
patience_counter = 0
best_state = None

for epoch in range(1, N_EPOCHS + 1):
    # -- Train
    model_nn.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()
        logits = model_nn(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimiser.step()
        epoch_loss += loss.item() * len(y_batch)
        print(f"\rEpoch {epoch:4d} | Batch loss: {loss.item():.5f}", end="")
    train_loss = epoch_loss / len(y_tr)

    # -- Evaluate
    model_nn.eval()
    with torch.no_grad():
        test_logits = model_nn(X_val.to(device))
        test_loss   = criterion(test_logits, y_val.to(device)).item()

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    scheduler.step(test_loss)

    if epoch % 1 == 0:
        print(f"Epoch {epoch:4d} | Train loss: {train_loss:.5f} | "
              f"Test loss: {test_loss:.5f}")

    # -- Early stopping
    if test_loss < best_test_loss:
        best_test_loss  = test_loss
        best_state      = {k: v.cpu().clone()
                           for k, v in model_nn.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(no improvement for {PATIENCE} epochs)")
            break

# Restore best weights
model_nn.load_state_dict(best_state)
print(f"\nBest test loss: {best_test_loss:.5f}")

        

In [ ]:
# ── Training curve ────────────────────────────────────────────────────────────
epochs_ran = len(train_losses)
fig, axes  = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(1, epochs_ran+1), train_losses,
        lw=2, color="steelblue", label="Train loss")
ax.plot(range(1, epochs_ran+1), test_losses,
        lw=2, color="tomato",    label="Test loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Training curve — full")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

zoom_start = int(0.8 * epochs_ran)
ax = axes[1]
ax.plot(range(zoom_start+1, epochs_ran+1), train_losses[zoom_start:],
        lw=2, color="steelblue", label="Train loss")
ax.plot(range(zoom_start+1, epochs_ran+1), test_losses[zoom_start:],
        lw=2, color="tomato",    label="Test loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Training curve — zoomed (last 20%)")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Gap diagnostic ────────────────────────────────────────────────────────────
final_gap = abs(train_losses[-1] - test_losses[-1])
print(f"Train/test loss gap at final epoch:  {final_gap:.6f}")
print(f"{'⚠ Possible overfitting' if final_gap > 0.05 else '✓ No significant overfitting detected'}")


# ── Save ──────────────────────────────────────────────────────────────────────
torch.save(best_state, "track_matching_nn.pt")
print("Model saved to track_matching_nn.pt")

In [ ]:
df_test = df_test.copy()


X_te = torch.tensor(X_test_scaled,  dtype=torch.float32)
y_te = torch.tensor(y_test.values,  dtype=torch.float32)

model_nn.eval()
with torch.no_grad():
    logits = model_nn(X_te.to(device)).cpu().numpy()

df_test["score"] = torch.sigmoid(torch.tensor(logits)).numpy()

METRICS = ["score"]

# CM

In [ ]:
importlib.reload(Evaluation)


df_leader_thresholded = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
# df_leader_thresholded = df_leader_thresholded[df_leader_thresholded["prob_True"] > 0.5]
# df_with_cut = df_test[(df_test[['prob_True','prob_Not True']] > 0.0).any(axis=1)]

df_cm = df_leader_thresholded

# Get predictions on test set
y_pred = df_cm["score"]
y_true = df_cm[TARGET]

Evaluation.cm(y_pred=y_pred,y_true=y_true, normalize='all')

# PR curve (Purity Efficiency)

In [ ]:
Evaluation.pr(y_pred = df_leader_thresholded['score'], y_true = df_leader_thresholded[TARGET])
Evaluation.pr(y_pred = df_test['score'], y_true = df_test[TARGET])

# Model Calibration curve and brier score

In [ ]:
from sklearn.calibration import calibration_curve

y_true = df_test['IsSignal']
y_prob = df_test['score']

prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=20, strategy='uniform')

# estimate bin counts and standard error
bins = np.linspace(0, 1, 21)
bin_idx = np.digitize(y_prob, bins) - 1
bin_counts = np.bincount(bin_idx, minlength=len(prob_true))
bin_positives = np.bincount(bin_idx, weights=y_true, minlength=len(prob_true))

# avoid divide-by-zero
mask = bin_counts > 0
stderr = np.zeros_like(prob_true)
stderr[mask] = 3*np.sqrt(prob_true[mask] * (1 - prob_true[mask]) / bin_counts[mask])
# Current error on N? sigma, to be investigated how the imbalance should be handled

disp = CalibrationDisplay.from_predictions(
    y_true=y_true,
    y_prob=y_prob,
    n_bins=20,
    name="Model",
)
ax = disp.ax_
ax.errorbar(prob_pred[mask], prob_true[mask], yerr=stderr[mask],
            fmt='.', color='black', capsize=3)
plt.show()

In [ ]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(
    y_true=df_test["IsSignal"],
    y_proba=df_test["score"]
)

print(f"Brier score: {brier:.6f}")

# Permutation feature importance

In [ ]:
def compute_permutation_importance(
    model, 
    X_val, 
    y_val, 
    features,
    n_repeats=10,
    random_state=42,
    n_jobs=5
):
    """
    Compute permutation importance and return as a sorted Series.
    
    Parameters
    ----------
    model : fitted XGBoost model
    X_val : validation feature matrix
    y_val : validation labels
    features : list of feature names
    n_repeats : number of times to permute (default 10)
    random_state : for reproducibility
    n_jobs : parallel jobs (-1 = all cores)
    
    Returns
    -------
    perm_df : DataFrame with importance, std, and percentile
    """
    
    perm_result = permutation_importance(
        model, 
        X_val, 
        y_val,
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=n_jobs
    )
    
    perm_df = pd.DataFrame({
        'feature': features,
        'perm_importance': perm_result.importances_mean,
        'std': perm_result.importances_std,
    }).sort_values('perm_importance', ascending=True)
    
    perm_df['importance_percentile'] = perm_df['perm_importance'].rank(pct=True)
    
    return perm_df


def plot_permutation_importance(
    perm_df,
    figsize=(10, 10),
    title="Permutation Importance (Validation Set)",
    show_threshold=None,
    threshold_label=None
):
    """
    Plot permutation importance with error bars.
    
    Parameters
    ----------
    perm_df : DataFrame from compute_permutation_importance
    figsize : figure size
    title : plot title
    show_threshold : optional importance threshold line to plot
    threshold_label : label for threshold line
    
    Returns
    -------
    fig, ax : matplotlib figure and axis
    """
    
    fig, ax = plt.subplots(figsize=figsize)
    
    y_pos = np.arange(len(perm_df))
    ax.barh(
        y_pos, 
        perm_df['perm_importance'],
        xerr=perm_df['std'],
        error_kw={'ecolor': 'gray', 'alpha': 0.5, 'capsize': 3},
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(perm_df['feature'])
    ax.set_xlabel("Permutation Importance (error bars = ±1 std)")
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.3)
    
    if show_threshold is not None:
        ax.axvline(
            show_threshold, 
            color='red', 
            linestyle='--', 
            linewidth=2,
            label=threshold_label or f'Threshold = {show_threshold:.4f}'
        )
        ax.legend()
    
    plt.tight_layout()
    
    return fig, ax


def compare_importances(
    model,
    X_val,
    y_val,
    features,
    figsize=(14, 10),
    n_repeats=10,
    random_state=42,
):
    """
    Compute and plot both native XGBoost importance and permutation importance side-by-side.
    
    Parameters
    ----------
    model : fitted XGBoost model
    X_val : validation features
    y_val : validation labels
    features : list of feature names
    figsize : figure size
    n_repeats : number of permutation repeats
    random_state : for reproducibility
    
    Returns
    -------
    perm_df : DataFrame with permutation importance
    fig, axes : matplotlib figure and axes
    """
    
    # Native importance
    native_imp = pd.Series(
        model.feature_importances_,
        index=features
    ).sort_values(ascending=True)
    
    # Permutation importance
    perm_df = compute_permutation_importance(
        model, X_val, y_val, features,
        n_repeats=n_repeats,
        random_state=random_state
    )
    
    # Create side-by-side plots
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Native importance (left)
    axes[0].barh(range(len(native_imp)), native_imp.values, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[0].set_yticks(range(len(native_imp)))
    axes[0].set_yticklabels(native_imp.index)
    axes[0].set_xlabel("Feature Importance (gain)")
    axes[0].set_title("XGBoost Native Importance")
    axes[0].grid(axis='x', alpha=0.3)
    
    # Permutation importance (right)
    y_pos = np.arange(len(perm_df))
    axes[1].barh(
        y_pos,
        perm_df['perm_importance'],
        xerr=perm_df['std'],
        error_kw={'ecolor': 'gray', 'alpha': 0.5, 'capsize': 3},
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(perm_df['feature'])
    axes[1].set_xlabel("Permutation Importance (error bars = ±1 std)")
    axes[1].set_title("Permutation Importance (Validation)")
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    
    return perm_df, fig, axes


# Example usage:
"""
# Assuming you have: model, X_val, y_val, FEATURES

# Option 1: Just permutation importance
perm_df = compute_permutation_importance(model, X_val, y_val, FEATURES)
fig, ax = plot_permutation_importance(perm_df)
plt.show()

# Option 2: Compare side-by-side with native importance
perm_df, fig, axes = compare_importances(model, X_val, y_val, FEATURES)
plt.show()

# Option 3: Apply filtering
perm_df_filtered = perm_df[perm_df['importance_percentile'] > 0.4]
fig, ax = plot_permutation_importance(
    perm_df_filtered,
    title="Permutation Importance (Top 60%)",
    show_threshold=perm_df[perm_df['importance_percentile'] == 0.4]['perm_importance'].values[0],
    threshold_label="40th percentile cutoff"
)
plt.show()

print(perm_df[['feature', 'perm_importance', 'std', 'importance_percentile']].to_string())
"""


# should this be uuh train or test?
perm_df = compute_permutation_importance(model, df_val[FEATURES], df_val[TARGET], FEATURES)
fig, ax = plot_permutation_importance(perm_df)
plt.show()

# perm_df, fig, axes = compare_importances(model, df_val[FEATURES], df_val[TARGET], FEATURES)
# plt.show()

In [ ]:
perm_df

In [ ]:
# # perm_df = compute_permutation_importance(model, X_test, y_test, FEATURES) unneeded as it is already calculated in the compare_importances function
# perm_df_filtered = perm_df[perm_df['importance_percentile'] > 0.4]
# fig, ax = plot_permutation_importance(
#     perm_df_filtered,
#     title="Permutation Importance (Top 60%)",
#     show_threshold=perm_df[perm_df['importance_percentile'] >= 0.4]['importance'].values[0],
#     threshold_label="40th percentile cutoff"
# )
# plt.show()

# print(perm_df[['feature', 'importance', 'std', 'importance_percentile']].to_string())

In [ ]:


# # Get native importance as DataFrame
# native_df = pd.DataFrame({
#     'feature': FEATURES,
#     'native_importance': model.feature_importances_
# })

# # Prepare permutation DataFrame
# perm_df_renamed = perm_df[['feature', 'importance']].rename(columns={'importance': 'perm_importance'})

# # Merge the two
# importance_comparison = pd.merge(native_df, perm_df_renamed, on='feature')

# # Normalize each importance metric to [0,1] by dividing by its maximum
# importance_comparison['native_importance_norm'] = importance_comparison['native_importance'] / importance_comparison['native_importance'].max()
# importance_comparison['perm_importance_norm'] = importance_comparison['perm_importance'] / importance_comparison['perm_importance'].max()

# # Compute difference (normalized permutation - normalized native)
# importance_comparison['difference'] = importance_comparison['perm_importance_norm'] - importance_comparison['native_importance_norm']

# # Sort by absolute difference for better visualization
# importance_comparison = importance_comparison.sort_values('difference', key=abs, ascending=False)

# # Plot
# fig, ax = plt.subplots(figsize=(12, 8))
# bars = ax.barh(importance_comparison['feature'], importance_comparison['difference'],
#                color=['red' if x < 0 else 'blue' for x in importance_comparison['difference']],
#                alpha=0.7)

# ax.set_xlabel('Normalized Difference (Permutation - Native Importance)')
# ax.set_title('Normalized Difference between Permutation and Native Feature Importance')
# ax.grid(axis='x', alpha=0.3)

# # Add value labels on bars
# for bar, diff in zip(bars, importance_comparison['difference']):
#     width = bar.get_width()
#     ax.text(width + (0.01 if width >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
#             f'{diff:.3f}', ha='left' if width >= 0 else 'right', va='center', fontsize=8)

# plt.tight_layout()
# plt.show()

# # Also print the comparison table
# print("Normalized Feature Importance Comparison:")
# print(importance_comparison[['feature', 'native_importance_norm', 'perm_importance_norm', 'difference']].round(4))

# Group-Level Evaluation

In [ ]:
df_test.head()

In [ ]:
METRICS = ["score"]

In [ ]:
# g = xgb.to_graphviz(
#     model,
#     num_trees=10,
#     graph_attr={"dpi": "300", "size": "20,10"}
# )

# g.render("xgb_tree", format="png", cleanup=True)

# All matches

In [ ]:
match_groups = Utils.build_match_groups(df_test)
Utils.draw_all_features(features=METRICS, match_groups=match_groups, density=False, log = True)

# Leading Match Metric Distribution Analysis

In [ ]:
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=METRICS, match_groups=match_groups_leader, density=False, per=0.0,log = True)

# Metric score sweep

In [ ]:
importlib.reload(Utils)

for entry in METRICS:
    Utils.sweep_threshold_plot(df_eval= df_test, metrics_fn = Utils.inhousemetrics, title=entry +" vs Score Threshold", score_col=entry, Nsigma=3.0, n_steps=100)

# Match Assigned Analysis

In [ ]:
# Apply threshold to get final matches, then plot feature distributions for the accepted candidates - see firsthand the contamination
threshold = 0.8
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
df_leader = df_leader[df_leader["score"] >= threshold].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=FEATURES, match_groups=match_groups_leader, density=True, per=0.01)

# Featurewise Metric breakdown

In [ ]:
importlib.reload(Utils)
mch_cols = ["PhiMCH", "TanlMCH", "InvQPtMCH", "PtMCH", "etaMCH"] # MCH cols we can actually use for these metrics as they require the underlying group structure to be preserved
common_cols= mch_cols + ['PDCA', 'Rabs', 'MFTMult']
#TODO: confirm these preserve grouping. Add non mch group preserving structures... like possibly MFTMult if it remains defined based on the best chi2 tracks around a mch track - correspond t
for entry in common_cols:   
    Utils.plot_metrics_vs_feature(df=df_test,feature=entry, threshold = 0.8, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=25, trim_low=0.0, trim_high=0., Nsigma=1.0)

# Delta Investigation

In [ ]:
df_group_eval = df_test.sort_values(["mchID", "score"], ascending=[True, False])

In [ ]:
df_group_eval["rank"] = df_group_eval.groupby("mchID").cumcount()

top1 = df_group_eval[df_group_eval["rank"] == 0][["mchID", "score"]].rename(columns={"score": "max_score"})
top2 = df_group_eval[df_group_eval["rank"] == 1][["mchID", "score"]].rename(columns={"score": "second_max_score"})

In [ ]:
group_stats = top1.merge(top2, on="mchID", how="left")
group_stats["second_max_score"] = group_stats["second_max_score"].fillna(0.0)

group_stats["delta_1_2"] = group_stats["max_score"] - group_stats["second_max_score"]

In [ ]:
agg_stats = df_group_eval.groupby("mchID")["score"].agg(["mean", "std"]).reset_index()
agg_stats = agg_stats.rename(columns={"mean": "mean_score", "std": "std_score"})

In [ ]:
labels = df_group_eval.groupby("mchID")["IsSignal"].max().reset_index()
labels = labels.rename(columns={"IsSignal": "has_true_match"})

In [ ]:
group_stats = group_stats.merge(agg_stats, on="mchID")
group_stats = group_stats.merge(labels, on="mchID")

In [ ]:
true_groups = group_stats[group_stats["has_true_match"] == 1]
false_groups = group_stats[group_stats["has_true_match"] == 0]

In [ ]:

plt.hist(true_groups["max_score"], bins=50, alpha=0.5, label="True-match groups")
plt.hist(false_groups["max_score"], bins=50, alpha=0.5, label="No-match groups")
plt.xlabel("Max score per group")
plt.ylabel("Count")
plt.legend()
plt.title("Group-level max score distribution")
plt.show()

In [ ]:
plt.hist(true_groups["delta_1_2"], bins=50, alpha=0.5, label="True-match groups")
plt.hist(false_groups["delta_1_2"], bins=50, alpha=0.5, label="No-match groups")
plt.xlabel("Score gap: top1 - top2")
plt.ylabel("Count")
plt.legend()
plt.title("Group-level score separation (delta)")
plt.show()

In [ ]:
plt.figure(figsize=(7,6))

# sns.kdeplot(
#     data=true_groups,
#     x="max_score",
#     y="mean_score",
#     cmap="Reds",
#     fill=True,
#     alpha=0.5,
#     label="True-match"
# )

# sns.kdeplot(
#     data=false_groups,
#     x="max_score",
#     y="mean_score",
#     cmap="Blues",
#     fill=True,
#     alpha=0.5,
#     label="No-match"
# )

# plt.title("Density: True vs No-match groups")
# plt.legend()
# plt.show()

In [ ]:
# plt.scatter(false_groups["mean_score"], false_groups["max_score"], alpha=0.3, label="No-match groups")
# plt.xlabel("Mean score")
# plt.ylabel("Max score")
# plt.legend()
# plt.title("Group score structure")
# plt.show()


# SHAP Addition

In [ ]:
from matplotlib import cm

def compute_shap_importance_scalable(
    model,
    X_data,
    feature_names=None,
    sample_size=None,
    top_k=25,
    viz_modes=['bar_top_k', 'beeswarm_top_k', 'dependence_grid'],
    **kwargs
):
    """
    Compute SHAP values and generate scalable visualizations for many features.
    
    Args:
        model: Trained XGBoost model
        X_data: feature matrix (np.array or pd.DataFrame)
        feature_names: List of feature names
        sample_size: Max samples for explainer (e.g., 5000 for speed)
        top_k: Number of top features to visualize in detail (default 25)
        viz_modes: List of visualizations to generate
                   Options: 'bar_all', 'bar_top_k', 'beeswarm_top_k', 
                           'dependence_grid', 'heatmap_top_k'
        **kwargs: Extra args for compatibility
    
    Returns:
        dict with SHAP values, explainer, importance dataframe, and vis objects
    """
    
    # Extract feature names and convert to array
    if feature_names is None:
        if isinstance(X_data, pd.DataFrame):
            feature_names = X_data.columns.tolist()
            X_array = X_data.values
        else:
            feature_names = [f"Feat_{i}" for i in range(X_data.shape[1])]
            X_array = X_data
    else:
        X_array = X_data.values if isinstance(X_data, pd.DataFrame) else X_data
    
    n_features = len(feature_names)
    print(f"\n{'='*60}")
    print(f"Computing SHAP for {n_features} features, {len(X_array)} samples")
    print(f"{'='*60}\n")
    
    # Subsample for explainer if needed
    if sample_size is not None and len(X_array) > sample_size:
        idx = np.random.choice(len(X_array), size=sample_size, replace=False)
        X_sample = X_array[idx]
        print(f"Using {sample_size} samples for explainer (subsampled from {len(X_array)})")
    else:
        X_sample = X_array
        sample_size = len(X_array)
    
    # Create explainer and compute SHAP values
    print("Computing TreeSHAP values...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # Handle multi-class
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    
    # Compute importance metrics
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'Mean |SHAP|': mean_abs_shap,
        'Rank': np.arange(1, n_features + 1)
    }).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)
    importance_df['Rank'] = np.arange(1, len(importance_df) + 1)
    
    print("\n=== Top 15 Features by Mean |SHAP| ===")
    print(importance_df.head(15)[['Rank', 'feature', 'Mean |SHAP|']].to_string(index=False))
    print(f"\n... and {n_features - 15} more features")
    
    # Visualizations
    print(f"\nGenerating visualizations: {viz_modes}")
    figs = {}
    
    if 'bar_all' in viz_modes:
        figs['bar_all'] = _plot_bar_all_features(importance_df)
    
    if 'bar_top_k' in viz_modes:
        figs['bar_top_k'] = _plot_bar_top_k(importance_df, top_k=top_k)
    
    if 'beeswarm_top_k' in viz_modes:
        figs['beeswarm_top_k'] = _plot_beeswarm_top_k(
            shap_values, X_sample, importance_df, feature_names, top_k=top_k
        )
    
    if 'dependence_grid' in viz_modes:
        figs['dependence_grid'] = _plot_dependence_grid(
            shap_values, X_sample, importance_df, feature_names, top_k=min(top_k, 12)
        )
    
    if 'heatmap_top_k' in viz_modes:
        figs['heatmap_top_k'] = _plot_heatmap_top_k(
            shap_values, importance_df, feature_names, top_k=top_k, n_samples_plot=100
        )
    
    return {
        'shap_values': shap_values,
        'explainer': explainer,
        'X_sample': X_sample,
        'importance_df': importance_df,
        'feature_names': feature_names,
        'mean_abs_shap': mean_abs_shap,
        'figs': figs
    }


def _plot_bar_all_features(importance_df, figsize=(14, 20)):
    """Bar plot of all features (tall, scrollable)."""
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = cm.RdYlGn_r(np.linspace(0.2, 0.8, len(importance_df)))
    ax.barh(importance_df['feature'], importance_df['Mean |SHAP|'], color=colors)
    ax.set_xlabel('Mean Absolute SHAP Value', fontsize=11)
    ax.set_title(f'SHAP Importance: All {len(importance_df)} Features', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    fig.tight_layout()
    return fig


def _plot_bar_top_k(importance_df, top_k=25, figsize=(10, 8)):
    """Bar plot of top-k features."""
    df_top = importance_df.head(top_k)
    
    fig, ax = plt.subplots(figsize=figsize)
    # colors = cm.viridis(np.linspace(0, 1, len(df_top)))
    colors = cm.RdYlGn_r(np.linspace(0.2, 0.8, len(importance_df)))
    ax.barh(df_top['feature'], df_top['Mean |SHAP|'], color=colors)
    ax.set_xlabel('Mean Absolute SHAP Value', fontsize=11)
    ax.set_title(f'Top {top_k} Features by SHAP Importance', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    fig.tight_layout()
    return fig


def _plot_beeswarm_top_k(shap_values, X_sample, importance_df, feature_names, top_k=25):
    """
    Beeswarm plot for top-k features.
    Each point = one sample. Color = feature value. Y-axis = SHAP value.
    Shows distribution and direction of SHAP contributions.
    """
    df_top = importance_df.head(top_k)
    top_indices = [feature_names.index(f) for f in df_top['feature']]
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    y_pos = 0
    y_labels = []
    y_ticks = []
    
    for rank, (feature, idx) in enumerate(zip(df_top['feature'], top_indices)):
        shap_vals = shap_values[:, idx]
        feature_vals = X_sample[:, idx]
        
        # Normalize feature values to [0, 1] for coloring
        fv_norm = (feature_vals - feature_vals.min()) / (feature_vals.max() - feature_vals.min() + 1e-8)
        colors = cm.coolwarm(fv_norm)
        
        # Add jitter to avoid overplotting
        y_jitter = np.random.normal(y_pos, 0.04, size=len(shap_vals))
        ax.scatter(shap_vals, y_jitter, c=colors, s=30, alpha=0.6, edgecolor='none')
        
        y_labels.append(f"#{rank+1}: {feature}")
        y_ticks.append(y_pos)
        y_pos += 1
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.set_xlabel('SHAP Value (← negative impact | positive impact →)', fontsize=11)
    ax.set_title(f'SHAP Beeswarm: Top {top_k} Features\n(Color: feature value, cold→hot)', 
                 fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()
    return fig


def _plot_dependence_grid(shap_values, X_sample, importance_df, feature_names, top_k=12):
    """
    Grid of dependence plots: SHAP value vs feature value for top-k features.
    Each subplot shows one feature.
    """
    df_top = importance_df.head(top_k)
    top_indices = [feature_names.index(f) for f in df_top['feature']]
    
    n_cols = 3
    n_rows = (len(df_top) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten()
    
    for idx, (feature, feat_idx) in enumerate(zip(df_top['feature'], top_indices)):
        ax = axes[idx]
        
        shap_vals = shap_values[:, feat_idx]
        feature_vals = X_sample[:, feat_idx]
        
        # Color by feature value
        sc = ax.scatter(feature_vals, shap_vals, c=feature_vals, cmap='coolwarm', 
                       s=40, alpha=0.6, edgecolor='none')
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_xlabel(feature, fontsize=10)
        ax.set_ylabel('SHAP Value', fontsize=10)
        ax.set_title(f'{feature}', fontsize=11, fontweight='bold')
        plt.colorbar(sc, ax=ax, label='feature Value')
        ax.grid(alpha=0.3)
    
    # Hide unused subplots
    for idx in range(len(df_top), len(axes)):
        axes[idx].axis('off')
    
    fig.suptitle(f'SHAP Dependence Plots: Top {top_k} Features', 
                fontsize=14, fontweight='bold', y=1.00)
    fig.tight_layout()
    return fig


def _plot_heatmap_top_k(shap_values, importance_df, feature_names, top_k=25, n_samples_plot=100):
    """
    Heatmap: samples × top-k features, with SHAP values as cell colors.
    Good for spotting patterns and interactions.
    """
    df_top = importance_df.head(top_k)
    top_indices = [feature_names.index(f) for f in df_top['feature']]
    
    # Subsample rows if too many
    if len(shap_values) > n_samples_plot:
        idx = np.random.choice(len(shap_values), size=n_samples_plot, replace=False)
        shap_sub = shap_values[idx]
    else:
        shap_sub = shap_values
    
    data_heat = shap_sub[:, top_indices]
    
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(data_heat, cmap='RdBu_r', aspect='auto', vmin=-np.abs(data_heat).max(), 
                   vmax=np.abs(data_heat).max())
    
    ax.set_yticks(np.arange(len(shap_sub)))
    ax.set_xticks(np.arange(len(df_top)))
    ax.set_xticklabels(df_top['feature'], rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels([f"Sample {i}" for i in range(len(shap_sub))], fontsize=8)
    
    cbar = plt.colorbar(im, ax=ax, label='SHAP Value')
    ax.set_title(f'SHAP Heatmap: Top {top_k} Features × {len(shap_sub)} Samples\n(Blue=negative, Red=positive)', 
                fontsize=13, fontweight='bold')
    fig.tight_layout()
    return fig

In [ ]:
results_shap = compute_shap_importance_scalable(
    model=model,
    X_data=df_val[FEATURES],
    feature_names=FEATURES,
    sample_size=25000,
    top_k=25,
    viz_modes=['bar_top_k', 'beeswarm_top_k', 'dependence_grid', 'heatmap_top_k']
)

# Access results
print(results_shap['importance_df'].head(20))

# Individual feature analysis
fig = results_shap['figs']['beeswarm_top_k']

In [ ]:
shap_importance_df = results_shap['importance_df']

In [ ]:
def compute_ks(df, feature, label_col="IsSignal"):
    sig = df[df[label_col] == 1][feature]
    bkg = df[df[label_col] == 0][feature]

    ks_stat, p_value = ks_2samp(sig, bkg)
    
    return ks_stat, p_value

def ks_scan(df, features, label_col="IsSignal"):
    results = []

    for feat in features:
        ks, p = compute_ks(df, feat, label_col)
        results.append({
            "feature": feat,
            "ks_stat": ks,
            "p_value": p
        })

    return pd.DataFrame(results).sort_values("ks_stat", ascending=False)

In [ ]:
df_ks = ks_scan(df, FEATURES)


# Importance aggregation

In [ ]:
# Simple snippet of code here that pools together:
# Permutation importance + KS + SHAP + XGB importance
# Averages over the ranking of each of these metrics to compute an aggregate average ranking, from which we can then choose the top X features for use

In [ ]:
importance_df = (
    importance_xgb_gain
    .merge(perm_df[['feature','perm_importance']], on="feature")
    .merge(shap_importance_df[['feature', 'Mean |SHAP|']], on="feature")
    .merge(df_ks[['feature','ks_stat']], on="feature")
)

In [ ]:
importance_df

In [ ]:
for col in ["xgb_gain", "perm_importance", 'Mean |SHAP|','ks_stat']:
    importance_df[f"{col}_rank"] = (
        importance_df[col]
        .rank(ascending=False)
    )

importance_df["mean_rank"] = (
    importance_df[[c for c in importance_df.columns if c.endswith("_rank")]]
    .mean(axis=1)
)

importance_df = importance_df.sort_values("mean_rank")

In [ ]:
importance_df


In [ ]:
FEATURES_ORDERED = importance_df['feature'].to_numpy()
FEATURES_ORDERED

In [ ]:
FEATURES_ORDERED_PRESERVED = ['DeltaDirection', 'PullPt', 'APullPhi', 'PullTanl', 'PtMFT',
       'CPhiPhiMFT', 'DeltaR', 'SameSign', 'DeltaEta', 'DeltaTanl',
       'RelPtDiff', 'CYYMFT', 'C1Pt1PtMFT', 'PullPhi', 'CXXMFT',
       'C1PtPhiMFT', 'YMCH', 'XMCH', 'DeltaPt', 'ADeltaPhi', 'PullR',
       'PullX', 'PullY', 'CXYMFT', 'CTglTglMCH', 'DeltaPhi', 'C1PtXMFT',
       'CPhiPhiMCH', 'InvQPtMFT', 'DeltaY', 'C1PtPhiMCH', 'APullX',
       'etaMCH', 'TanlMCH', 'ADeltaY', 'TanlMFT', 'DeltaX', 'APullY',
       'ADeltaX', 'etaMFT', 'C1PtYMFT', 'CTglTglMFT', 'XMFT', 'CYYMCH',
       'CPhiXMFT', 'C1PtTglMFT', 'YMFT', 'CPhiXMCH', 'CXXMCH', 'CPhiYMCH',
       'PhiMCH', 'TrackTypeMFT', 'CTglXMFT', 'C1Pt1PtMCH', 'CTglYMFT',
       'PhiMFT', 'CPhiYMFT', 'CTglPhiMFT', 'CTglYMCH', 'CXYMCH',
       'CTglXMCH', 'PtMCH', 'CTglPhiMCH', 'C1PtTglMCH', 'C1PtXMCH',
       'C1PtYMCH', 'InvQPtMCH']

PbPb_feature_importance = ['RelPtDiff', 'DeltaDirection', 'SameSign', 'DeltaR', 'PtMFT',
       'ADeltaPhi', 'CXXMFT', 'CPhiPhiMCH', 'PullPt', 'CYYMFT',
       'C1Pt1PtMFT', 'CPhiPhiMFT', 'CTglTglMCH', 'C1Pt1PtMCH', 'DeltaPt',
       'TanlMFT', 'PullR', 'ADeltaX', 'DeltaTanl', 'PullTanl', 'ADeltaY',
       'C1PtPhiMFT', 'CTglTglMFT', 'etaMFT', 'DeltaEta', 'APullPhi',
       'CXXMCH', 'XMCH', 'CTglXMCH', 'YMCH', 'PtMCH', 'CXYMFT', 'CYYMCH',
       'C1PtXMFT', 'InvQPtMFT', 'DeltaPhi', 'DeltaX', 'DeltaY', 'PullX',
       'APullX', 'CPhiYMFT', 'C1PtYMFT', 'CTglXMFT', 'CPhiXMFT', 'PullY',
       'XMFT', 'APullY', 'YMFT', 'TanlMCH', 'PhiMFT', 'etaMCH',
       'CTglPhiMFT', 'PullPhi', 'CTglYMFT', 'CPhiXMCH', 'CTglYMCH',
       'CPhiYMCH', 'TrackTypeMFT', 'PhiMCH', 'CTglPhiMCH', 'C1PtXMCH',
       'C1PtTglMCH', 'C1PtTglMFT', 'C1PtPhiMCH', 'CXYMCH', 'C1PtYMCH',
       'InvQPtMCH']

# Feature Ablation Curve

In [ ]:
# feature_counts = (
#     list(range(1,11))
#     + [15,20,25,30,50,67]
# )

# results = []

# for n in feature_counts:

#     selected_features = PbPb_feature_importance[:n]

#     model = xgb.XGBClassifier(
#         objective="binary:logistic",
#         **optuna_params,
#         eval_metric="aucpr",
#         early_stopping_rounds=3,
#         random_state=42,
#         n_jobs=-1,
#         tree_method="hist",
#     )

#     model.fit(
#         df_train[selected_features],
#         df_train[TARGET],
#         eval_set=[
#             (df_val[selected_features], df_val[TARGET])
#         ],
#         verbose=False,
#     )

#     scores = model.predict_proba(
#         df_val[selected_features]
#     )[:,1]

#     results.append({
#         "n_features": n,
#         "prauc": average_precision_score(
#             df_val[TARGET],
#             scores
#         ),
#         "roc_auc": roc_auc_score(
#             df_val[TARGET],
#             scores
#         ),
#         "logloss": log_loss(
#             df_val[TARGET],
#             scores
#         ),
#         "best_iteration": model.best_iteration,
#     })
#     print('finished model with: ' + str(n) +' featrures')

# results_df = pd.DataFrame(results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    results_df["n_features"],
    results_df["prauc"],
    marker="o",
    label="PR AUC"
)

ax.set_xlabel("Number of features")
ax.set_ylabel("PR AUC")
ax.set_title("Feature Count Study")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 8)
)

# PRAUC
axes[0,0].plot(
    results_df["n_features"],
    results_df["prauc"],
    marker="o"
)
axes[0,0].set_title("PR AUC")

# ROC AUC
axes[0,1].plot(
    results_df["n_features"],
    results_df["roc_auc"],
    marker="o"
)
axes[0,1].set_title("ROC AUC")

# LogLoss
axes[1,0].plot(
    results_df["n_features"],
    results_df["logloss"],
    marker="o"
)
axes[1,0].set_title("Log Loss")

# Best iteration
axes[1,1].plot(
    results_df["n_features"],
    results_df["best_iteration"],
    marker="o"
)
axes[1,1].set_title("Best Iteration")

for ax in axes.flat:
    ax.set_xlabel("Number of features")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
ax.set_ylim(
    results_df["prauc"].min() * 0.999,
    results_df["prauc"].max() * 1.001
)

In [ ]:
full_prauc = results_df["prauc"].iloc[-1]

results_df["prauc_fraction"] = (
    results_df["prauc"] / full_prauc
)

fig, ax = plt.subplots(figsize=(8,5))

ax.plot(
    results_df["n_features"],
    100 * results_df["prauc_fraction"],
    marker="o"
)

ax.grid(True)
ax.set_xlabel("Number of features")
ax.set_ylabel("% of full-model PRAUC")
ax.set_title("Performance Retention")

plt.tight_layout()
plt.show()

In [ ]:
results_df

# Uniqueness checks

In [ ]:
# grouped = df.groupby("mchID")

# # number of unique values per group, per column
# nunique_per_group = grouped.nunique()

# # columns where ALL groups have exactly 1 unique value
# constant_cols = nunique_per_group.eq(1).all(axis=0)

# # get column names
# constant_cols = constant_cols[constant_cols].index.tolist()
# print(constant_cols)

In [ ]:
# def plot_nunique_histograms(df, group_col="mchID", cols=None, bins=20, decimals=6):
#     import matplotlib.pyplot as plt

#     if cols is None:
#         cols = [c for c in df.columns if c != group_col]

#     # apply consistent rounding
#     df_rounded = df.copy()
#     df_rounded[cols] = df_rounded[cols].round(decimals)

#     nunique = df_rounded.groupby(group_col)[cols].nunique(dropna=False)

#     for col in cols:
#         values = nunique[col]

#         plt.figure()
#         plt.hist(values, bins=bins)
#         plt.title(f"{col} — nunique per {group_col} (rounded to {decimals})")
#         plt.xlabel("nunique within group")
#         plt.ylabel("count of groups")
#         plt.grid(True)

#         print(f"{col}: min={values.min()}, max={values.max()}, "
#               f"counts={values.value_counts().sort_index().to_dict()}")

#         plt.show()

In [ ]:
# plot_nunique_histograms(df, group_col="mchID", cols=FEATURES, bins=5)

# ONNX

In [ ]:
import sklearn

In [ ]:
model_original = sklearn.base.clone(model)

In [ ]:
model.get_booster().feature_names = [f"f{i}" for i in range(len(FEATURES))]

In [ ]:
import onnx
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as ort



In [ ]:
print(xgb.__version__)
print(onnx.__version__)

In [ ]:
initial_types = [('float_input', FloatTensorType([-1, len(FEATURES)]))]
onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_types)
onnxmltools.utils.save_model(onnx_model, 'model_PbPb.onnx')

In [ ]:
X_test_numpy = df[FEATURES].to_numpy(dtype=np.float32)

In [ ]:
# xgb_pred = model.predict_proba(X_test_numpy)[:, 1]
# print(f"XGBoost prediction: {xgb_pred}") 

In [ ]:
# sess = ort.InferenceSession("model.onnx")
# input_name = sess.get_inputs()[0].name
# onnx_label, onnx_prob = sess.run(None, {input_name: X_test_numpy})
# print(onnx_prob)

In [ ]:
# for o in sess.get_outputs():
#     print(o.name, o.shape, o.type)

In [ ]:
# print(sess.get_inputs()[0])

In [ ]:
# diff = np.abs(xgb_pred - onnx_prob[:, 1])  # Assuming onnx_pred is a list of outputs and the probabilities are in the first output
# print("Max abs diff:", np.max(diff))
# print("Mean abs diff:", np.mean(diff))

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_true=df_test['IsSignal'],y_score= df_test['score'])
print(auc)

# Debug

In [ ]:
df_test[['IsSignal','score']].head()

In [ ]:
df_test["score"].describe()

In [ ]:
model

In [ ]:
model_original.predict_proba(df_val[FEATURES])[:, 1]

FAT MODEL CLASSIFICATION REPORT - using all trainable features we obtain:
Classification Report:

              precision    recall  f1-score   support
    Not True       1.00      0.98      0.99   1123132
        True       0.89      0.99      0.94    234538

    accuracy                           0.98   1357670
   macro avg       0.95      0.98      0.96   1357670
weighted avg       0.98      0.98      0.98   1357670


Overall Accuracy: 0.9782

Per-class Performance:
  Not True - Precision: 0.9982, Recall: 0.9755, F1: 0.9867
  True     - Precision: 0.8942, Recall: 0.9914, F1: 0.9403

SLIM MODEL CLASSIFICATION REPORT - using the top 13 trainable features we obtain:

Classification Report:
              precision    recall  f1-score   support

    Not True       1.00      0.96      0.98   1123132
        True       0.83      0.99      0.90    234538

    accuracy                           0.96   1357670
   macro avg       0.91      0.97      0.94   1357670
weighted avg       0.97      0.96      0.96   1357670


Overall Accuracy: 0.9632

Per-class Performance:
  Not True - Precision: 0.9857, Recall: 0.9584, F1: 0.9718
  True     - Precision: 0.9595, Recall: 0.9861, F1: 0.9726
